# Spatial Placement Model — Training

Dataset: MUSDB18 stems with pan labels derived from L/R energy ratio in the mix.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from pathlib import Path
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score

In [ ]:
df = pd.read_csv('../data/spatial_dataset.csv')
print(f'Total samples: {len(df)}')
print(f'Stems: {df["stem"].value_counts().to_dict()}')
print(f'Pan range: {df["pan"].min():.3f} to {df["pan"].max():.3f}')
print(f'Pan std: {df["pan"].std():.3f}')
df.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['pan'], bins=50, color='#6366f1', alpha=0.8)
axes[0].axvline(0, color='white', linewidth=1, linestyle='--', alpha=0.5)
axes[0].set_title('Pan distribution')
axes[0].set_xlabel('Pan (-1=left, +1=right)')
axes[0].set_facecolor('#0d0e18')
axes[0].figure.set_facecolor('#08090e')

for stem in df['stem'].unique():
    subset = df[df['stem'] == stem]['pan']
    axes[1].hist(subset, bins=30, alpha=0.6, label=stem)
axes[1].set_title('Pan by stem type')
axes[1].legend()
axes[1].set_facecolor('#0d0e18')

plt.tight_layout()
plt.savefig('../data/pan_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
FEATURE_COLS = [
    'spectral_centroid', 'spectral_bandwidth', 'spectral_rolloff',
    'rms', 'zcr', 'onset_rate',
    *[f'mfcc_{i}' for i in range(13)]
]

X = df[FEATURE_COLS].values
y = df['pan'].values

print(f'Features: {X.shape[1]}')
print(f'Samples: {X.shape[0]}')
print(f'Any NaN: {np.isnan(X).any()}')

In [ ]:
tracks = df['track'].unique()
np.random.seed(42)
np.random.shuffle(tracks)
split = int(len(tracks) * 0.8)
train_tracks = set(tracks[:split])
test_tracks = set(tracks[split:])

train_mask = df['track'].isin(train_tracks)
test_mask = df['track'].isin(test_tracks)

X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[test_mask], y[test_mask]

print(f'Train: {len(X_train)} samples from {len(train_tracks)} tracks')
print(f'Test:  {len(X_test)} samples from {len(test_tracks)} tracks')

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
model = MLPRegressor(
    hidden_layer_sizes=(128, 64, 32),
    activation='relu',
    solver='adam',
    learning_rate_init=0.001,
    max_iter=500,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=20,
    random_state=42,
    verbose=True
)

model.fit(X_train_scaled, y_train)
print(f'Converged after {model.n_iter_} iterations')

In [ ]:
y_pred = model.predict(X_test_scaled)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f'MAE:  {mae:.4f}  (pan units, range -1 to +1)')
print(f'R²:   {r2:.4f}')
print(f'Baseline MAE (predict mean): {mean_absolute_error(y_test, np.full_like(y_test, y_train.mean())):.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(y_test, y_pred, alpha=0.4, s=20, color='#6366f1')
axes[0].plot([-1, 1], [-1, 1], 'r--', linewidth=1)
axes[0].set_xlabel('True pan')
axes[0].set_ylabel('Predicted pan')
axes[0].set_title(f'Predicted vs True (R²={r2:.3f})')
axes[0].set_facecolor('#0d0e18')

residuals = y_pred - y_test
axes[1].hist(residuals, bins=50, color='#6366f1', alpha=0.8)
axes[1].axvline(0, color='red', linewidth=1, linestyle='--')
axes[1].set_title(f'Residuals (MAE={mae:.4f})')
axes[1].set_xlabel('Prediction error')
axes[1].set_facecolor('#0d0e18')

plt.tight_layout()
plt.savefig('../data/model_evaluation.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
baseline_mae = mean_absolute_error(y_test, model.predict(X_test_scaled))
importances = []

for i, feat in enumerate(FEATURE_COLS):
    X_perm = X_test_scaled.copy()
    np.random.shuffle(X_perm[:, i])
    perm_mae = mean_absolute_error(y_test, model.predict(X_perm))
    importances.append((feat, perm_mae - baseline_mae))

importances.sort(key=lambda x: x[1], reverse=True)
print('Feature importances (MAE increase when permuted):')
for feat, imp in importances[:10]:
    print(f'  {feat:<25} {imp:.5f}')

In [ ]:
out_dir = Path('../mixer/models')
out_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(model, out_dir / 'spatial_model.pkl')
joblib.dump(scaler, out_dir / 'spatial_scaler.pkl')

print(f'Model saved to {out_dir / "spatial_model.pkl"}')
print(f'Scaler saved to {out_dir / "spatial_scaler.pkl"}')

loaded_model = joblib.load(out_dir / 'spatial_model.pkl')
loaded_scaler = joblib.load(out_dir / 'spatial_scaler.pkl')
check = loaded_model.predict(loaded_scaler.transform(X_test_scaled[:5]))
print(f'Reload check — first 5 predictions: {check.round(3)}')